# Module 5 — Inverse Kinematics
## Unit 4 — From Geometry to Numerical IK
### Lesson 4.2 — The Local Linear Map: the FK Jacobian (for Solving)

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
The 2x2 Jacobian; its columns are per-joint gripper motions.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(theta1, theta2, L1, L2):
    x = L1*np.cos(theta1) + L2*np.cos(theta1+theta2)
    y = L1*np.sin(theta1) + L2*np.sin(theta1+theta2)
    return np.array([x, y])

def ik_2link_closed(x, y, L1, L2):
    """Closed-form: return both (theta1,theta2); [] unreachable; one on boundary."""
    c2 = (x*x + y*y - L1*L1 - L2*L2)/(2*L1*L2)
    if c2 < -1-1e-9 or c2 > 1+1e-9:
        return []
    c2 = max(-1.0, min(1.0, c2))
    sols=[]
    for sign in (+1.0, -1.0):
        s2 = sign*np.sqrt(max(0.0, 1.0 - c2*c2))
        t2 = np.arctan2(s2, c2)
        t1 = np.arctan2(y, x) - np.arctan2(L2*np.sin(t2), L1 + L2*np.cos(t2))
        sols.append((t1, t2))
        if abs(s2) < 1e-6:
            break
    return sols

def jacobian_2link(theta1, theta2, L1, L2):
    s1, s12 = np.sin(theta1), np.sin(theta1+theta2)
    c1, c12 = np.cos(theta1), np.cos(theta1+theta2)
    return np.array([[-L1*s1 - L2*s12, -L2*s12],
                     [ L1*c1 + L2*c12,  L2*c12]])

L1,L2=0.4,0.3
J=jacobian_2link(np.radians(30),np.radians(60),L1,L2)
print("J at (30,60) deg =\n",np.round(J,4))
print("column 2 (move theta2):",np.round(J[:,1],4))


## Coding Exercise (§8)
Finite-difference check: analytic columns match perturbation.


In [ ]:
# YOUR CODE HERE
def fd_jac(t1,t2,L1,L2,eps=1e-6):
    base=fk_two_link(t1,t2,L1,L2)
    c1=(fk_two_link(t1+eps,t2,L1,L2)-base)/eps
    c2=(fk_two_link(t1,t2+eps,L1,L2)-base)/eps
    return np.column_stack([c1,c2])
t1,t2=np.radians(30),np.radians(60)
assert np.allclose(jacobian_2link(t1,t2,L1,L2), fd_jac(t1,t2,L1,L2), atol=1e-4)
print("finite-difference check OK")


## Check your work


In [ ]:
J=jacobian_2link(np.radians(30),np.radians(60),L1,L2)
assert np.allclose(J,[[-0.5,-0.3],[0.3464,0.0]],atol=1e-3)
print("All checks passed.")
